# HorrorPlay · El Huésped — restauración experimental de la foto de Thomas y Nat

Notebook para Google Colab (GPU). Obtiene los JPG originales desde el commit verificado de la rama reboot/web-el-huesped, usa un modelo abierto de inpainting ejecutado **localmente en la GPU de Colab** y produce candidatos sin modificar el juego.

**Objetivo artístico:** pasar del aspecto de maniquíes 3D a una fotografía de albúmina victoriana creíble, conservando composición, contacto de la mano del mayor sobre el hombro del menor, contexto 1866 y la raspadura del capítulo 3.

**No utiliza APIs pagas ni otros modelos por suscripción.** Descargar pesos abiertos desde Hugging Face no es una llamada a una API de generación. El tiempo de GPU sí consume las unidades de Colab incluidas en tu plan Google AI Pro. Los cupos y GPU disponibles dependen de Google.

**Uso:** Entorno de ejecución → Cambiar tipo de entorno → GPU T4 o superior. Ejecutar las celdas en orden. Examinar la vista previa de la máscara antes de ejecutar la generación. No se hará push, commit ni sobrescritura de archivos de producción.

Modelos: SD 1.5 Inpainting por defecto (ligero para T4); SDXL Inpainting es opcional si la GPU y las unidades disponibles lo justifican. Las licencias de los pesos se deben revisar antes de publicar resultados comerciales.

In [ ]:
%pip -q install "diffusers>=0.35,<0.40" "transformers>=4.48,<5" "accelerate>=1.0" "safetensors>=0.5" "Pillow>=10" "requests>=2.31"

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter, ImageOps
from IPython.display import display
import base64, io, json, math, random, hashlib, requests
import torch

assert torch.cuda.is_available(), "Activa GPU: Entorno de ejecución > Cambiar tipo de entorno > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
ROOT = Path("/content/horrorplay_photo_experiment")
ROOT.mkdir(exist_ok=True, parents=True)
SOURCE_COMMIT = "58b69bd9c4df132ecd093ce02f89dcf1a7e89360"
REPO = "gabsvm/HorrorPlay"

def github_asset(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": SOURCE_COMMIT}, headers={"Accept": "application/vnd.github+json"}, timeout=45)
    r.raise_for_status()
    obj = r.json()
    assert obj.get("encoding") == "base64", f"No se recibio contenido base64: {path}"
    return base64.b64decode(obj["content"])

def load_source(path):
    raw = github_asset(path)
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    return image, hashlib.sha256(raw).hexdigest()

ORIGINAL, ORIGINAL_SHA = load_source("web/assets/closeups/room_photo.jpg")
TORN_ORIGINAL, TORN_SHA = load_source("web/assets/closeups/room_photo_torn.jpg")
assert ORIGINAL.size == (1280, 720), f"Dimensiones inesperadas: {ORIGINAL.size}"
assert TORN_ORIGINAL.size == ORIGINAL.size
print("Fuente intacta:", ORIGINAL.size, "sha256:", ORIGINAL_SHA[:16])
print("Fuente raspada: sha256:", TORN_SHA[:16])
display(ORIGINAL.resize((640, 360)))

## 1. Máscara de trabajo (revisar antes de gastar GPU)

La foto tiene anclas históricas invariables: cara del mayor (540,253) y del menor (672,297). Trabajamos solo sobre las figuras; el marco, firma E. VANCE y demás contenido se preservan.

La máscara roja muestra el área que la IA puede redibujar. Si su cobertura no te convence, **ajusta los puntos** y vuelve a ejecutarla antes de descargar ningún modelo.

In [ ]:
# ROI de 512x512 alineada: suficiente para las dos figuras.
LEFT, TOP, SIDE = 352, 96, 512
CROP_BOX = (LEFT, TOP, LEFT + SIDE, TOP + SIDE)
original_crop = ORIGINAL.crop(CROP_BOX)

# Polígono sobre cabezas, hombros, brazo compartido y vestimenta.
# Coordenadas EN EL LIENZO 1280x720 (no del recorte).
figure_polygon = [
    (471, 216), (523, 203), (571, 207), (607, 247),
    (647, 236), (696, 243), (731, 277), (757, 383),
    (765, 562), (451, 562), (445, 369), (452, 273)
]
full_mask = Image.new("L", ORIGINAL.size, 0)
d = ImageDraw.Draw(full_mask)
d.polygon(figure_polygon, fill=255)
mask_crop = full_mask.crop(CROP_BOX)  # blanco = zona editable
blend_mask = mask_crop.filter(ImageFilter.GaussianBlur(radius=14))

preview = ORIGINAL.copy().convert("RGBA")
overlay = Image.new("RGBA", ORIGINAL.size, (205, 34, 34, 0))
overlay.putalpha(full_mask.point(lambda v: int(v * 0.42)))
preview = Image.alpha_composite(preview, overlay)
display(preview.resize((640, 360)))
display(mask_crop.resize((384, 384)))
assert mask_crop.getbbox() is not None
assert mask_crop.getpixel((540-LEFT, 253-TOP)) == 255
assert mask_crop.getpixel((672-LEFT, 297-TOP)) == 255
print("Si la máscara está bien, continúa.")

## 2. Modelo y parámetros

Usa SD 1.5 para la primera pasada, incluso con Google AI Pro. Es una prueba controlada y consume menos recursos. Podrás cambiar MODEL_KIND a sdxl para otra tanda. El enlace al modelo se descarga desde Hugging Face, sin tokens de pago.

La IA podría cambiar rostros, edades o la pose. Los archivos generados son **candidatos**, no deben integrarse al juego automáticamente.

In [ ]:
# Opciones: "sd15" (T4, recomendado) o "sdxl" (más calidad potencial, más VRAM/tiempo)
MODEL_KIND = "sd15"
SEEDS = [1888, 1911, 1926]
MODEL_ID = {
    "sd15": "stable-diffusion-v1-5/stable-diffusion-inpainting",
    "sdxl": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
}[MODEL_KIND]

PROMPT = (
    "Authentic Victorian London 1866 albumen photographic studio portrait, two real human brothers, "
    "an older serious boy standing close to a younger boy, older brother's right hand rests naturally on "
    "younger brother's shoulder, composed formal restrained expressions, correct child anatomy, "
    "tailored modest period jackets, shirt collars, historically accurate Victorian clothing, "
    "soft sepia monochrome albumen paper grain, gentle photographic softness, realistic subtle faces, "
    "soft studio light, aged archival photograph, knee-length composition, genuine historical photograph"
)
NEGATIVE = (
    "3d render, CGI, game character, doll, mannequin, toy, puppet, figurine, plastic skin, wax figure, "
    "cartoon, oversharpened, modern suit, adult men, big shiny anime eyes, doll eyes, extra arms, "
    "extra fingers, malformed hands, disconnected hand, smiling teeth, duplicated people, typography, watermark"
)

from diffusers import AutoPipelineForInpainting
pipe = AutoPipelineForInpainting.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, use_safetensors=True
)
# Offload ayuda incluso si el runtime asigna una GPU de poca VRAM.
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print("Modelo cargado:", MODEL_ID)

## 3. Generar A/B/C y recuperar la variante raspada

Se mantiene la resolución original 1280×720. Solo se fusiona el área de las figuras mediante una máscara con borde suave. El resto del JPG se conserva en memoria píxel a píxel. La raspadura usa las anclas y el patrón determinista del generador original de la rama.

In [ ]:
def repaint_scrape(intact):
    """Raspadura original centrada en ELDER_HEAD; la cara de Nat permanece intacta."""
    im = intact.copy()
    d = ImageDraw.Draw(im, "RGBA")
    cx, cy, rx, ry = (540, 253, 46, 54)
    r = random.Random(1888)
    pts = []
    for k in range(26):
        a = k / 26 * 2 * math.pi
        wob = 1 + r.uniform(-0.22, 0.22)
        pts.append((cx + math.cos(a) * rx * 1.35 * wob,
                    cy + math.sin(a) * ry * 1.45 * wob))
    d.polygon(pts, fill=(201, 192, 174, 255))
    for _ in range(60):
        x, y = r.uniform(cx-rx, cx+rx), r.uniform(cy-ry, cy+ry)
        d.point((x, y), fill=(214, 206, 188, 255))
    for i in range(4):
        x0 = cx - rx * 0.7 + i * (rx * 0.5)
        d.line([x0, cy-ry*1.3, x0+rx*0.35, cy+ry*1.3], fill=(122,112,96,255), width=5)
        d.line([x0-2, cy-ry*1.3, x0+rx*0.35-2, cy+ry*1.3], fill=(228,220,202,255), width=2)
    return im.convert("RGB")

def make_candidate(seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    result = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        image=original_crop,
        mask_image=mask_crop,
        width=SIDE, height=SIDE,
        guidance_scale=7.0,
        num_inference_steps=32,
        strength=0.62,
        generator=generator
    ).images[0].convert("RGB")
    patch = Image.composite(result, original_crop, blend_mask)
    full = ORIGINAL.copy()
    full.paste(patch, (LEFT, TOP))
    return full

candidates = {}
for seed in SEEDS:
    intact = make_candidate(seed)
    torn = repaint_scrape(intact)
    assert intact.size == TORN_ORIGINAL.size
    candidates[seed] = (intact, torn)
    intact.save(ROOT / f"room_photo_candidate_{seed}.png")
    intact.save(ROOT / f"room_photo_candidate_{seed}.jpg", quality=92, subsampling=0)
    torn.save(ROOT / f"room_photo_torn_candidate_{seed}.jpg", quality=92, subsampling=0)
    print("Candidato", seed, "guardado")

print("Ningún archivo del repositorio fue modificado.")

## 4. Comparar a escala de juego y descargar un ZIP

Elige visualmente, no por cantidad de detalle. La prioridad es que ambos se reconozcan como **niños humanos reales** sin perder la mano sobre el hombro; después vendrá la validación dentro del motor WebGL a 640×360 y Android físico. Esta notebook no integra nada a producción.

In [ ]:
from PIL import ImageFont
source_rows = [("Actual", ORIGINAL, TORN_ORIGINAL)]
source_rows += [(f"Seed {seed}", *candidates[seed]) for seed in SEEDS]
w,h = 640,360
sheet = Image.new("RGB", (w*2, len(source_rows)*(h+34)), (19,19,21))
draw = ImageDraw.Draw(sheet)
for i, (label, intact, torn) in enumerate(source_rows):
    y = i * (h+34)
    sheet.paste(intact.resize((w,h), Image.Resampling.LANCZOS), (0,y+34))
    sheet.paste(torn.resize((w,h), Image.Resampling.LANCZOS), (w,y+34))
    draw.text((12,y+10), label + " — intacta", fill="white")
    draw.text((w+12,y+10), label + " — raspada", fill="white")
sheet.save(ROOT / "AB_photo_640.png")
display(sheet)
metadata = {
    "source_repo": REPO, "branch": "reboot/web-el-huesped",
    "source_commit": SOURCE_COMMIT, "source_sha256": ORIGINAL_SHA,
    "source_torn_sha256": TORN_SHA, "model": MODEL_ID, "seeds": SEEDS,
    "canvas": ORIGINAL.size, "crop": CROP_BOX,
    "mask": figure_polygon,
    "status": "EXPERIMENTAL — requires in-game and Android QA",
    "license_note": "Review OpenRAIL model terms before reuse",
}
(ROOT/"experiment.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
import shutil
archive = shutil.make_archive("/content/HorrorPlay_Thomas_Nat_Colab", "zip", ROOT)
print("ZIP:", archive)
from google.colab import files
files.download(archive)

## Integración posterior (no automatizada)

Para incorporar una variante tras aprobarla: sustituir en una rama de trabajo los dos archivos web/assets/closeups/room_photo.jpg y room_photo_torn.jpg usando el MISMO seed; comprobar el sprite de pared web/assets/props/room_photo.png para evitar desajustes; preservar hotspot, sello E. VANCE y anclas; ejecutar node web/tools/test-all.mjs; capturar habitación e inspección intacta/raspada en 640×360 y comprobar rendimiento en el Moto g86 Power. No mezclar resultados distintos entre capítulos.